In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import matplotlib.pyplot as plt


In [2]:
from torch.utils.data import Dataset as TorchDataset
import json
import os
import torch
import numpy as np
from pathlib import Path

class RawTokenDataset(TorchDataset):
    """ Loads raw uint32 tokens as memmap-backed array """
    def __init__(
        self,
        data_dir,
        window_size,
        stride=1,
        filter_interrupts=True,
        filter_overlaps=False
    ):
        """
        Args:
            data_dir: directory with the same format as `data/train_v0` and `data/val_v0`.
                Notably, has `video.bin` and `metadata.json`
            window_size: number of frames per "video" sequence
            stride: frame skip
            filter_interrupts: Under 3% of training frame sequences are the concatenation of two different clips.
                If filter_interrupts is True, will filter out these sequences using the segment ids.
            filter_overlaps: If False (default), one frame will appear in multiple examples;
                e.g. frame 0 might appear as the first frame in example 0 and also the second frame in example 15.
                If True, will filter out examples so that each frame appears at most once in the dataset.
        """
        data_dir = Path(data_dir)
        with open(data_dir / "metadata.json") as f:
            self.metadata = json.load(f)

        shape = (self.metadata["num_images"], self.metadata["s"], self.metadata["s"])
        video_tokens_path, segment_ids_path, action_tokens_path = [data_dir / f"{name}.bin"
                                                                   for name in ["video", "segment_ids", "actions"]]
        token_dtype = np.dtype(self.metadata.get("token_dtype", "uint32"))
        self.data = np.memmap(video_tokens_path, dtype=token_dtype, mode="r", shape=shape)
        # self.actions = np.memmap(action_tokens_path, dtype=np.uint16, mode="r", shape=(self.metadata["num_images"],))

        if os.path.isfile(segment_ids_path):
            self.segment_ids = np.memmap(
                segment_ids_path,
                dtype=np.int32,
                mode="r",
                shape=(self.metadata["num_images"],)
            )
        else:
            self.segment_ids = None
            if filter_interrupts:
                raise NotImplementedError("Cannot filter interrupted sequences without segment ids.")

        self.window_size, self.stride = window_size, stride
        # Number of frames between the first and last frames of a video sequence (excluding one endpoint frame)
        self.video_len = (self.window_size - 1) * self.stride

        self.valid_start_inds = []
        for start_ind in range(len(self.data) - self.video_len):
            # Assuming `segment_ids` is monotonically increasing, a sequence is interrupted
            # if the first and last frames have different segment ids.
            if not (filter_interrupts and self.segment_ids[start_ind] != self.segment_ids[start_ind + self.video_len]):
                self.valid_start_inds.append(start_ind)

        if filter_overlaps:
            # Instead of using a sliding window, use each frame at most once
            filtered_start_inds = []
            for start_ind in self.valid_start_inds:
                overlapping_start_inds = {start_ind - i * self.stride for i in range(1, self.window_size)}
                # all sequences from `overlapping_start_inds` will also contain `start_ind`,
                # so exclude sequence starting from `start_ind` if any of `overlapping_start_inds` is already being used
                for existing_start_ind in filtered_start_inds[-self.window_size * self.stride:]:
                    # Bound could be improved
                    if existing_start_ind in overlapping_start_inds:
                        break
                else:
                    filtered_start_inds.append(start_ind)

            self.valid_start_inds = filtered_start_inds

    def __len__(self):
        return len(self.valid_start_inds)

    def __getitem__(self, idx):
        """
        Returns a flattened sequence of tokens representing `self.window_size` frames,
        spaced `self.stride` apart.
        """
        start_ind = self.valid_start_inds[idx]
        x = torch.from_numpy((self.data[start_ind : start_ind + self.video_len + 1 : self.stride]).astype(np.int64))
        x = x.flatten()

        attention_mask = torch.ones_like(x)
        return {
            "input_ids": x,
            "labels": x,
            "attention_mask": attention_mask,
        }

In [3]:
class Cfg:
    data_dir = "./data/train_v0"
    window_size = 6
    stride = 1

    # slots
    num_slots = 8
    d_model = 128
    slot_iters = 5

    # decoder
    dec_hidden = 256

    # training
    batch_size = 8
    accum_steps = 8          # 実効バッチ = 64
    num_epochs = 20
    lr = 3e-4
    weight_decay = 1e-4
    grad_clip = 1.0

    # adaptive softmax
    cutoffs = [2000, 10000, 50000]   # 262144 に対して妥当な切り方（調整可）
    div_value = 4.0

    device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Cfg()


In [4]:
class SlotAttention(nn.Module):
    def __init__(self, num_slots, dim, iters=3):
        super().__init__()
        self.num_slots = num_slots
        self.iters = iters
        self.scale = dim ** -0.5

        self.slots_mu = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.slots_logsigma = nn.Parameter(torch.zeros(1, 1, dim))

        self.norm_in = nn.LayerNorm(dim)
        self.norm_slots = nn.LayerNorm(dim)
        self.norm_ff = nn.LayerNorm(dim)

        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)

        self.gru = nn.GRUCell(dim, dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim*2), nn.GELU(), nn.Linear(dim*2, dim)
        )

    def forward(self, inputs, prev_slots=None):
        # inputs: [B, N, D]
        B, N, D = inputs.shape

        if prev_slots is None:
            mu = self.slots_mu.expand(B, self.num_slots, -1)
            sigma = self.slots_logsigma.expand(B, self.num_slots, -1).exp()
            slots = mu + sigma * torch.randn_like(mu)
        else:
            slots = prev_slots

        x = self.norm_in(inputs)
        k = self.to_k(x)  # [B,N,D]
        v = self.to_v(x)

        for _ in range(self.iters):
            slots_prev = slots
            s = self.norm_slots(slots)
            q = self.to_q(s)  # [B,K,D]

            dots = torch.einsum("bkd,bnd->bkn", q, k) * self.scale  # [B,K,N]
            attn = dots.softmax(dim=1) + 1e-8                       # slot方向
            attn = attn / attn.sum(dim=-1, keepdim=True)            # N方向に正規化
            updates = torch.einsum("bnd,bkn->bkd", v, attn)         # [B,K,D]

            slots = self.gru(
                updates.reshape(-1, D),
                slots_prev.reshape(-1, D)
            ).reshape(B, self.num_slots, D)

            slots = slots + self.mlp(self.norm_ff(slots))

        return slots

class SlotTokenDecoderNoFullVocab(nn.Module):
    def __init__(self, num_slots, d_model, n_positions, dec_hidden):
        super().__init__()
        self.num_slots = num_slots
        self.n_positions = n_positions

        self.pos = nn.Parameter(torch.randn(1, 1, n_positions, d_model) * 0.02)

        self.mlp = nn.Sequential(
            nn.Linear(d_model, dec_hidden), nn.GELU(),
            nn.Linear(dec_hidden, d_model), nn.GELU(),
        )
        self.mask_head = nn.Linear(d_model, 1)

    def forward(self, slots):
        # slots: [B,K,D] -> hidden: [B,K,N,D], mask_logits: [B,K,N]
        B, K, D = slots.shape
        N = self.n_positions
        h = slots[:, :, None, :].expand(B, K, N, D) + self.pos
        h = self.mlp(h)
        mask_logits = self.mask_head(h).squeeze(-1)  # [B,K,N]
        return h, mask_logits



In [5]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast

class DiscreteTokenSlotModelAdaptive(nn.Module):
    def __init__(self, s, window_size, vocab_size, num_slots, d_model, slot_iters, dec_hidden,
                 cutoffs, div_value):
        super().__init__()
        self.s = s
        self.T = window_size
        self.N = s * s
        self.V = vocab_size
        self.K = num_slots
        self.D = d_model

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_time = nn.Parameter(torch.randn(1, window_size, 1, d_model) * 0.02)
        self.pos_space = nn.Parameter(torch.randn(1, 1, self.N, d_model) * 0.02)

        self.slot_attn = SlotAttention(num_slots=num_slots, dim=d_model, iters=slot_iters)
        self.decoder = SlotTokenDecoderNoFullVocab(num_slots, d_model, self.N, dec_hidden)

        self.adaptive = nn.AdaptiveLogSoftmaxWithLoss(
            in_features=d_model,
            n_classes=vocab_size,
            cutoffs=cutoffs,
            div_value=div_value
        )

    def forward(
        self,
        tokens,                      # [B,T,N]
        detach_slots=True,           # ★これがOOM回避の要
        pos_sample=None,             # 例: 64 or 128 (Noneなら全256)
        return_assign=False
    ):
        B, T, N = tokens.shape
        assert T == self.T and N == self.N

        # embed
        x = self.tok_emb(tokens) + self.pos_time[:, :T] + self.pos_space  # [B,T,N,D]

        total_nll = 0.0
        prev_slots = None
        hard_list = [] if return_assign else None

        # 位置サブサンプリングのインデックス（バッチ共通）
        if pos_sample is not None and pos_sample < N:
            pos_idx = torch.randperm(N, device=tokens.device)[:pos_sample]
        else:
            pos_idx = None

        for t in range(T):
            inp = x[:, t]  # [B,N,D]
            if pos_idx is not None:
                inp_sa = inp[:, pos_idx]          # [B,P,D]
                y = tokens[:, t, pos_idx]         # [B,P]
                P = pos_idx.numel()
            else:
                inp_sa = inp                       # [B,N,D]
                y = tokens[:, t]                   # [B,N]
                P = N

            # slot inference
            slots = self.slot_attn(inp_sa, prev_slots=prev_slots)  # [B,K,D]
            prev_slots = slots.detach() if detach_slots else slots

            # decode only on sampled positions: decoder側の pos を部分参照する必要があるので工夫
            # ここは簡易に、decoderが全N想定なので、pos_idxがある場合は decoder.pos も同じindexを取る
            h, mask_logits = self.decoder(slots)  # h: [B,K,N,D], mask_logits: [B,K,N]
            if pos_idx is not None:
                h = h[:, :, pos_idx, :]           # [B,K,P,D]
                mask_logits = mask_logits[:, :, pos_idx]  # [B,K,P]

            log_pi = F.log_softmax(mask_logits, dim=1)  # [B,K,P]

            # adaptive log prob for targets (avoid full vocab logits)
            h_flat = h.reshape(B * self.K * P, self.D)
            y_rep = y[:, None, :].expand(B, self.K, P).reshape(-1)

            # adaptive は fp32 計算が安全（内部で大きい行列）
            with autocast(enabled=False):
                out = self.adaptive(h_flat.float(), y_rep)
                log_pk_y = out.output.view(B, self.K, P).to(h.dtype)  # [B,K,P]

            logp = torch.logsumexp(log_pi + log_pk_y, dim=1)  # [B,P]
            nll = -logp.mean()
            total_nll = total_nll + nll

            if return_assign:
                masks = torch.softmax(mask_logits, dim=1)   # [B,K,P]
                hard = torch.argmax(masks, dim=1)           # [B,P]
                # 表示のため、P点だけでも良いが、N全体に戻したいなら別途対応
                hard_list.append(hard)

        loss = total_nll / T
        if return_assign:
            hard = torch.stack(hard_list, dim=1)  # [B,T,P]
            return loss, hard
        return loss


In [6]:
# --- あなたの RawTokenDataset をここで使う ---
# dataset = RawTokenDataset(cfg.data_dir, cfg.window_size, stride=cfg.stride, ...)
cfg.data_dir = "/root/work/data/raw/train_v1.1"  # ★ここを自分のパスに
dataset = RawTokenDataset(
    data_dir=cfg.data_dir,
    window_size=cfg.window_size,
    stride=cfg.stride,
    filter_interrupts=True,
    filter_overlaps=False,
)

s = dataset.metadata["s"]
vocab_size = int(dataset.metadata["vocab_size"])  # 262144
T = cfg.window_size
N = s * s

model = DiscreteTokenSlotModelAdaptive(
    s=s, window_size=T, vocab_size=vocab_size,
    num_slots=cfg.num_slots, d_model=cfg.d_model, slot_iters=cfg.slot_iters,
    dec_hidden=cfg.dec_hidden, cutoffs=cfg.cutoffs, div_value=cfg.div_value
).to(cfg.device)

# 例: cfg.batch_size=8, cfg.accum_steps=8 推奨
loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True, drop_last=True, num_workers=0)

opt = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = GradScaler()

s = dataset.metadata["s"]
T = cfg.window_size
N = s*s

history = []
global_step = 0
opt.zero_grad(set_to_none=True)

for epoch in range(cfg.num_epochs):
    model.train()
    total = 0.0
    pbar = tqdm(loader, desc=f"epoch {epoch+1}/{cfg.num_epochs}")

    for batch in pbar:
        x = batch["input_ids"].to(cfg.device)  # [B, L]
        # ★固定cfg.batch_sizeではなく x.shape[0] を使う
        B = x.shape[0]
        x = x.view(B, T, N).long()

        with autocast():
            loss = model(
                x,
                detach_slots=True,      # ★これが最重要
                pos_sample=64,          # ★まず64で。安定したら128/Noneへ
                return_assign=False
            )
            loss = loss / cfg.accum_steps

        scaler.scale(loss).backward()
        global_step += 1

        if global_step % cfg.accum_steps == 0:
            scaler.unscale_(opt)
            if cfg.grad_clip and cfg.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

        loss_val = float(loss.item() * cfg.accum_steps)
        total += loss_val
        pbar.set_postfix(nll=f"{loss_val:.4f}")

    avg = total / len(loader)
    history.append(avg)
    print(f"Epoch {epoch+1}: avg NLL = {avg:.4f}")


/tmp/ipykernel_97646/852198010.py:27: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
epoch 1/20:   0%|          | 0/1343665 [00:00<?, ?it/s]/tmp/ipykernel_97646/852198010.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_97646/756126576.py:84: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
epoch 1/20:   0%|          | 25/1343665 [09:44<8725:48:27, 23.38s/it, nll=19.2943] 


KeyboardInterrupt: 

In [ ]:
@torch.no_grad()
def visualize_hard_assign(hard, s, max_frames=6):
    # hard: [T,N]
    T = hard.shape[0]
    Tshow = min(T, max_frames)
    plt.figure(figsize=(2*Tshow, 2))
    for t in range(Tshow):
        plt.subplot(1, Tshow, t+1)
        plt.imshow(hard[t].view(s, s).cpu().numpy())
        plt.axis("off")
        plt.title(f"t={t}")
    plt.show()

# 1バッチだけ確認
model.eval()
batch = next(iter(loader))
x = batch["input_ids"].to(cfg.device).view(cfg.batch_size, T, N).long()
loss, hard = model(x, return_assign=True)
print("loss:", float(loss.item()))
visualize_hard_assign(hard[0].cpu(), s=s, max_frames=T)


In [14]:
## cudaメモリ解放
torch.cuda.empty_cache()
